In [1]:
from langgraph.graph import StateGraph,START,END
# from typing import TypedDict,Literal, Annotated
# from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

In [23]:
import langchain
import operator
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver 

In [3]:
model=ChatGroq(model='llama-3.1-8b-instant',api_key='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')

In [4]:
class Chatbot(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [5]:
def chat_model(state: Chatbot):
    # take user query
    message = state["messages"]
    # send to llm
    response = model.invoke(message)
    #  generate response
    return {"messages": [response]}

In [25]:
check_pointer = MemorySaver()

graph =StateGraph(Chatbot)

# add node 
graph.add_node("chat_model", chat_model)

# add_edges
graph.add_edge(START, "chat_model")
graph.add_edge("chat_model", END)

chatbot = graph.compile(checkpointer=check_pointer)

In [13]:
initial_state = {
    "messages": [HumanMessage(content= "what is the capital of pakistan")]
}
chatbot.invoke(initial_state)["messages"][-1]

AIMessage(content='The capital of Pakistan is Islamabad.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 41, 'total_tokens': 49, 'completion_time': 0.006630794, 'completion_tokens_details': None, 'prompt_time': 0.001956964, 'prompt_tokens_details': None, 'queue_time': 0.005326864, 'total_time': 0.008587758}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b452c-cf16-74a3-814a-d8004b0c0a5f-0', usage_metadata={'input_tokens': 41, 'output_tokens': 8, 'total_tokens': 49})

In [29]:
thread_id = "1"
while True:
    user_input = input("User Type")
    print("User messages\n")
    if user_input.lower() in ["exit", "bye"]:
        break
    config = {"configurable": {'thread_id': thread_id}}

        
    response = chatbot.invoke({"messages":HumanMessage(content= user_input)}, config=config)
    print("Ai",response["messages"][-1].content )

User Type hy


User messages

Ai Hello. Is there something I can help you with, or would you like to chat?


User Type my name is usman


User messages

Ai Nice to meet you, Usman. How's your day going so far?


User Type can you tell my name?


...

User messages

Ai 10 + 100 = 110


User Type and multiply 2 of my adding result


User messages

Ai You want me to multiply 2 by 110. 

2 * 110 = 220


User Type exit


User messages



In [31]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hy', additional_kwargs={}, response_metadata={}, id='54511aa1-c404-42a3-ad36-90ecb6f85ce8'), AIMessage(content='Hello. Is there something I can help you with, or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 36, 'total_tokens': 55, 'completion_time': 0.028740459, 'completion_tokens_details': None, 'prompt_time': 0.001613242, 'prompt_tokens_details': None, 'queue_time': 0.005382063, 'total_time': 0.030353701}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b45ca-d329-7ad1-af00-d65f51395654-0', usage_metadata={'input_tokens': 36, 'output_tokens': 19, 'total_tokens': 55}), HumanMessage(content='my name is usman', additional_kwargs={}, response_metadata={}, id='cd7ac0dc-db58-453d-82a1-7b46ddc2489b'), AIMessage(cont